In [1]:
import pyspark.sql as sql
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark : sql.SparkSession = (
    sql.SparkSession.builder
	.appName("etl_operational_points")
	.getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/04 13:02:01 WARN Utils: Your hostname, BEBEL-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/04 13:02:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/04 13:02:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Data extraction

In [2]:
df : sql.DataFrame = spark.read.csv("data/operational_points.csv", header=True, sep=";", inferSchema=True)
df.show(5, truncate=False)

+-------------------------------------+---------------------------------------------------------------------------------+--------+------------+--------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+
|Geo Point                            |Geo Shape                                                                        |PTCAR ID|Code TAF/TAP|Nom symbolique|Abréviation LST FR courte|Abréviation LST NL courte|Abréviation LST FR complète|Abréviation LST NL complète|Nom FR court|Nom NL court|Nom FR moyen       |Nom NL moyen       |Nom FR complet     |Nom NL complet     |Classification NL|Classification EN   |Classification FR      |
+-------------------------------------+---------------------------------------------------------------------------------+-------

In [3]:
df.printSchema()

root
 |-- Geo Point: string (nullable = true)
 |-- Geo Shape: string (nullable = true)
 |-- PTCAR ID: integer (nullable = true)
 |-- Code TAF/TAP: string (nullable = true)
 |-- Nom symbolique: string (nullable = true)
 |-- Abréviation LST FR courte: string (nullable = true)
 |-- Abréviation LST NL courte: string (nullable = true)
 |-- Abréviation LST FR complète: string (nullable = true)
 |-- Abréviation LST NL complète: string (nullable = true)
 |-- Nom FR court: string (nullable = true)
 |-- Nom NL court: string (nullable = true)
 |-- Nom FR moyen: string (nullable = true)
 |-- Nom NL moyen: string (nullable = true)
 |-- Nom FR complet: string (nullable = true)
 |-- Nom NL complet: string (nullable = true)
 |-- Classification NL: string (nullable = true)
 |-- Classification EN: string (nullable = true)
 |-- Classification FR: string (nullable = true)



# Data validation

## Null values

In [4]:
null_counts = []
for column in df.columns:
    null_count = df.filter(F.col(column).isNull()).count()
    null_counts.append((column, null_count))
for col_name, null_count in null_counts:
    print(f"Column '{col_name}' has {null_count} null values")

Column 'Geo Point' has 17 null values
Column 'Geo Shape' has 17 null values
Column 'PTCAR ID' has 0 null values
Column 'Code TAF/TAP' has 0 null values
Column 'Nom symbolique' has 0 null values
Column 'Abréviation LST FR courte' has 0 null values
Column 'Abréviation LST NL courte' has 0 null values
Column 'Abréviation LST FR complète' has 0 null values
Column 'Abréviation LST NL complète' has 0 null values
Column 'Nom FR court' has 0 null values
Column 'Nom NL court' has 0 null values
Column 'Nom FR moyen' has 0 null values
Column 'Nom NL moyen' has 0 null values
Column 'Nom FR complet' has 0 null values
Column 'Nom NL complet' has 0 null values
Column 'Classification NL' has 0 null values
Column 'Classification EN' has 0 null values
Column 'Classification FR' has 0 null values


## Unique values

In [5]:
unique_counts = []
for column in df.columns :
    unique_count : int = df.agg(F.count_distinct(column).alias('count')).collect()[0]['count']
    unique_counts.append((column, unique_count))
for col_name, unique_count in unique_counts :
    print(f"Column {col_name} has {unique_count} unique values.")

Column Geo Point has 1320 unique values.
Column Geo Shape has 1320 unique values.
Column PTCAR ID has 1337 unique values.
Column Code TAF/TAP has 1337 unique values.
Column Nom symbolique has 1337 unique values.
Column Abréviation LST FR courte has 1337 unique values.
Column Abréviation LST NL courte has 1337 unique values.
Column Abréviation LST FR complète has 1337 unique values.
Column Abréviation LST NL complète has 1337 unique values.
Column Nom FR court has 1337 unique values.
Column Nom NL court has 1337 unique values.
Column Nom FR moyen has 1337 unique values.
Column Nom NL moyen has 1337 unique values.
Column Nom FR complet has 1337 unique values.
Column Nom NL complet has 1337 unique values.
Column Classification NL has 12 unique values.
Column Classification EN has 12 unique values.
Column Classification FR has 12 unique values.


## Empty values

In [6]:
empty_counts = []
string_columns = [f.name for f in df.schema.fields if isinstance(f.dataType, T.StringType)]
for column in string_columns:
    empty_count = df.filter(F.col(column) == "").count()
    empty_counts.append((column, empty_count))
for col_name, empty_count in empty_counts:
    print(f"Column '{col_name}' has {empty_count} empty values")

Column 'Geo Point' has 0 empty values
Column 'Geo Shape' has 0 empty values
Column 'Code TAF/TAP' has 0 empty values
Column 'Nom symbolique' has 0 empty values
Column 'Abréviation LST FR courte' has 0 empty values
Column 'Abréviation LST NL courte' has 0 empty values
Column 'Abréviation LST FR complète' has 0 empty values
Column 'Abréviation LST NL complète' has 0 empty values
Column 'Nom FR court' has 0 empty values
Column 'Nom NL court' has 0 empty values
Column 'Nom FR moyen' has 0 empty values
Column 'Nom NL moyen' has 0 empty values
Column 'Nom FR complet' has 0 empty values
Column 'Nom NL complet' has 0 empty values
Column 'Classification NL' has 0 empty values
Column 'Classification EN' has 0 empty values
Column 'Classification FR' has 0 empty values


## Duplicate values

In [7]:
duplicates = df.groupBy(df.columns).count().filter(F.col("count") > 1)
duplicate_count = duplicates.count()
if duplicate_count > 0:
    print(f"There are {duplicate_count} duplicate lines in the data")
else:
    print("No duplicate lines found in the data")

No duplicate lines found in the data


# Data transformation

## Converting and renaming the column "Geo Point"

In [8]:
def truncate(value : float, decimals : int = 6) :
    if value is None :
        return None
    return round(value, decimals)

truncate_udf = F.udf(truncate, T.DoubleType())

new_df = df.withColumn(
	"Geo_x",
	truncate_udf(F.split(F.col("Geo Point"), ",")[0].cast("double"))
).withColumn(
	"Geo_y",
	truncate_udf(F.split(F.col("Geo Point"), ",")[1].cast("double"))
).drop("Geo Point")
new_df.show(5, truncate=False)

+---------------------------------------------------------------------------------+--------+------------+--------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|Geo Shape                                                                        |PTCAR ID|Code TAF/TAP|Nom symbolique|Abréviation LST FR courte|Abréviation LST NL courte|Abréviation LST FR complète|Abréviation LST NL complète|Nom FR court|Nom NL court|Nom FR moyen       |Nom NL moyen       |Nom FR complet     |Nom NL complet     |Classification NL|Classification EN   |Classification FR      |Geo_x    |Geo_y   |
+---------------------------------------------------------------------------------+--------+------------+--------------+-------------------------+--------------------

## Dropping the column "Geo shape"

In [9]:
new_df = new_df.drop("Geo Shape")
new_df.show(5, truncate=False)

+--------+------------+--------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|PTCAR ID|Code TAF/TAP|Nom symbolique|Abréviation LST FR courte|Abréviation LST NL courte|Abréviation LST FR complète|Abréviation LST NL complète|Nom FR court|Nom NL court|Nom FR moyen       |Nom NL moyen       |Nom FR complet     |Nom NL complet     |Classification NL|Classification EN   |Classification FR      |Geo_x    |Geo_y   |
+--------+------------+--------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+-----

## Converting and renaming the column "PTCAR ID"

In [10]:
new_df = new_df.withColumn("PTCAR ID", F.col("PTCAR ID").cast("int")).withColumnRenamed("PTCAR ID", "ID")
new_df.show(5, truncate=False)

+----+------------+--------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|ID  |Code TAF/TAP|Nom symbolique|Abréviation LST FR courte|Abréviation LST NL courte|Abréviation LST FR complète|Abréviation LST NL complète|Nom FR court|Nom NL court|Nom FR moyen       |Nom NL moyen       |Nom FR complet     |Nom NL complet     |Classification NL|Classification EN   |Classification FR      |Geo_x    |Geo_y   |
+----+------------+--------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|443 |B

## Renaming the column "Code TAF/TAP"

In [11]:
new_df = new_df.withColumnRenamed("Code TAF/TAP", "Code_TAF_TAP")
new_df.show(5, truncate=False)

+----+------------+--------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|ID  |Code_TAF_TAP|Nom symbolique|Abréviation LST FR courte|Abréviation LST NL courte|Abréviation LST FR complète|Abréviation LST NL complète|Nom FR court|Nom NL court|Nom FR moyen       |Nom NL moyen       |Nom FR complet     |Nom NL complet     |Classification NL|Classification EN   |Classification FR      |Geo_x    |Geo_y   |
+----+------------+--------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|443 |B

## Renaming the column "Nom symbolique"

In [12]:
new_df = new_df.withColumnRenamed("Nom symbolique", "Symbolic_name")
new_df.show(5, truncate=False)

+----+------------+-------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|ID  |Code_TAF_TAP|Symbolic_name|Abréviation LST FR courte|Abréviation LST NL courte|Abréviation LST FR complète|Abréviation LST NL complète|Nom FR court|Nom NL court|Nom FR moyen       |Nom NL moyen       |Nom FR complet     |Nom NL complet     |Classification NL|Classification EN   |Classification FR      |Geo_x    |Geo_y   |
+----+------------+-------------+-------------------------+-------------------------+---------------------------+---------------------------+------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|443 |BE00

## Dropping unecessary name columns

In [13]:
new_df = new_df.drop("Abréviation LST FR courte", "Abréviation LST NL courte", "Abréviation LST FR complète", 
    "Abréviation LST NL complète", "Nom FR moyen", "Nom NL moyen"
)
new_df.show(5, truncate=False)

+----+------------+-------------+------------+------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|ID  |Code_TAF_TAP|Symbolic_name|Nom FR court|Nom NL court|Nom FR complet     |Nom NL complet     |Classification NL|Classification EN   |Classification FR      |Geo_x    |Geo_y   |
+----+------------+-------------+------------+------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|443 |BE00443     |FKGLF        |Genk-Zd-Ford|Genk-Zd-Ford|Genk-Zuid-L.O.-Ford|Genk-Zuid-L.O.-Ford|Station          |Station             |Gare                   |50.923363|5.497283|
|275 |BE00275     |GCRA         |Charleroi-AT|Charleroi-AT|Charleroi-A.T.     |Charleroi-A.T.     |Dienstinstallatie|Service installation|Installation de service|50.400574|4.458827|
|1748|BE01748     |VOIL1        |V.Oiltank1  |V.Oiltank1  |Verb.Oiltanking 1  |Verb.Oiltan

## Renaming the name columns

In [14]:
new_df = new_df.withColumnRenamed(
	"Nom FR court",
    "Name_FR_short"
).withColumnRenamed(
    "Nom NL court",
    "Name_NL_short"
).withColumnRenamed(
    "Nom FR complet",
    "Name_FR_full"
).withColumnRenamed(
    "Nom NL complet",
    "Name_NL_full"
)
new_df.show(5, truncate=False)

+----+------------+-------------+-------------+-------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|ID  |Code_TAF_TAP|Symbolic_name|Name_FR_short|Name_NL_short|Name_FR_full       |Name_NL_full       |Classification NL|Classification EN   |Classification FR      |Geo_x    |Geo_y   |
+----+------------+-------------+-------------+-------------+-------------------+-------------------+-----------------+--------------------+-----------------------+---------+--------+
|443 |BE00443     |FKGLF        |Genk-Zd-Ford |Genk-Zd-Ford |Genk-Zuid-L.O.-Ford|Genk-Zuid-L.O.-Ford|Station          |Station             |Gare                   |50.923363|5.497283|
|275 |BE00275     |GCRA         |Charleroi-AT |Charleroi-AT |Charleroi-A.T.     |Charleroi-A.T.     |Dienstinstallatie|Service installation|Installation de service|50.400574|4.458827|
|1748|BE01748     |VOIL1        |V.Oiltank1   |V.Oiltank1   |Verb.Oiltanking 1  

## Dropping unecessary classification columns

In [15]:
new_df = new_df.drop("Classification NL", "Classification FR")
new_df.show(5, truncate=False)

+----+------------+-------------+-------------+-------------+-------------------+-------------------+--------------------+---------+--------+
|ID  |Code_TAF_TAP|Symbolic_name|Name_FR_short|Name_NL_short|Name_FR_full       |Name_NL_full       |Classification EN   |Geo_x    |Geo_y   |
+----+------------+-------------+-------------+-------------+-------------------+-------------------+--------------------+---------+--------+
|443 |BE00443     |FKGLF        |Genk-Zd-Ford |Genk-Zd-Ford |Genk-Zuid-L.O.-Ford|Genk-Zuid-L.O.-Ford|Station             |50.923363|5.497283|
|275 |BE00275     |GCRA         |Charleroi-AT |Charleroi-AT |Charleroi-A.T.     |Charleroi-A.T.     |Service installation|50.400574|4.458827|
|1748|BE01748     |VOIL1        |V.Oiltank1   |V.Oiltank1   |Verb.Oiltanking 1  |Verb.Oiltanking 1  |Connection          |51.306684|4.293916|
|1749|BE01749     |VOIL2        |V.Oiltank2   |V.Oiltank2   |Verb.Oiltanking 2  |Verb.Oiltanking 2  |Connection          |51.308302|4.290241|
|1751|

## Renaming the column "Classification EN"

In [16]:
new_df = new_df.withColumnRenamed("Classification EN", "Classification")
new_df.show(5, truncate=False)

+----+------------+-------------+-------------+-------------+-------------------+-------------------+--------------------+---------+--------+
|ID  |Code_TAF_TAP|Symbolic_name|Name_FR_short|Name_NL_short|Name_FR_full       |Name_NL_full       |Classification      |Geo_x    |Geo_y   |
+----+------------+-------------+-------------+-------------+-------------------+-------------------+--------------------+---------+--------+
|443 |BE00443     |FKGLF        |Genk-Zd-Ford |Genk-Zd-Ford |Genk-Zuid-L.O.-Ford|Genk-Zuid-L.O.-Ford|Station             |50.923363|5.497283|
|275 |BE00275     |GCRA         |Charleroi-AT |Charleroi-AT |Charleroi-A.T.     |Charleroi-A.T.     |Service installation|50.400574|4.458827|
|1748|BE01748     |VOIL1        |V.Oiltank1   |V.Oiltank1   |Verb.Oiltanking 1  |Verb.Oiltanking 1  |Connection          |51.306684|4.293916|
|1749|BE01749     |VOIL2        |V.Oiltank2   |V.Oiltank2   |Verb.Oiltanking 2  |Verb.Oiltanking 2  |Connection          |51.308302|4.290241|
|1751|

## Reordering the columns

In [ ]:
final_df : sql.DataFrame = new_df.select(
    "ID",
    "Geo_x",
    "Geo_y",
    "Code_TAF_TAP",
    "Classification",
    "Symbolic_name",
    "Name_FR_short",
    "Name_FR_full",
)
final_df.show(5, truncate=False)

+----+---------+--------+------------+--------------------+-------------+-------------+-------------------+-------------+-------------------+
|ID  |Geo_x    |Geo_y   |Code_TAF_TAP|Classification      |Symbolic_name|Name_FR_short|Name_FR_full       |Name_NL_short|Name_NL_full       |
+----+---------+--------+------------+--------------------+-------------+-------------+-------------------+-------------+-------------------+
|443 |50.923363|5.497283|BE00443     |Station             |FKGLF        |Genk-Zd-Ford |Genk-Zuid-L.O.-Ford|Genk-Zd-Ford |Genk-Zuid-L.O.-Ford|
|275 |50.400574|4.458827|BE00275     |Service installation|GCRA         |Charleroi-AT |Charleroi-A.T.     |Charleroi-AT |Charleroi-A.T.     |
|1748|51.306684|4.293916|BE01748     |Connection          |VOIL1        |V.Oiltank1   |Verb.Oiltanking 1  |V.Oiltank1   |Verb.Oiltanking 1  |
|1749|51.308302|4.290241|BE01749     |Connection          |VOIL2        |V.Oiltank2   |Verb.Oiltanking 2  |V.Oiltank2   |Verb.Oiltanking 2  |
|1751|

In [18]:
final_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Code_TAF_TAP: string (nullable = true)
 |-- Classification: string (nullable = true)
 |-- Symbolic_name: string (nullable = true)
 |-- Name_FR_short: string (nullable = true)
 |-- Name_FR_full: string (nullable = true)
 |-- Name_NL_short: string (nullable = true)
 |-- Name_NL_full: string (nullable = true)



# Data loading

In [19]:
final_df.coalesce(1).write.mode("overwrite").option("header", True).csv("cleaned_data/operational_points", sep=";")